# Import Statements

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import yfinance as yf

from modules.utils import *
from modules.screen import *
from modules.data_loader import *
from modules.portfolio import *
from modules.trade import *
from modules.backtest import *
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from modules.tearsheet import Tearsheet

import plotly.io as pio
pio.renderers.default = "notebook_connected"

---
# Parameters

In [2]:
config = load_config("config.yaml")

# Extract config parameters
start_date = config["start_date"]
end_date = config["end_date"]
BENCHMARK = config["benchmark"]
UPDATE = config["update"]
SCREEN = config["screen"]

---
# Data Fetching

In [3]:
# Loading data for all NSE tickers and calculating features
full_nse_tickers = load_nse_all()
load_full_data = load_data(start_date=start_date, end_date=end_date, update=UPDATE, full_nse_tickers=full_nse_tickers,
                           benchmark=BENCHMARK)
# load benchmark data
benchmark_prices = download_benchmark_data(symbol=BENCHMARK)

# Forward fill missing price data for each ticker
df = load_full_data.copy().reset_index()
df["Date"] = pd.to_datetime(df["Date"])

priceData = (
    df.pivot(index="Date", columns="TIC", values=["Adj Close", "Open"])
      .sort_index()
      .ffill()
)

priceData = (
    priceData
    .stack(level=1)
    .reset_index()
)
priceData.columns = ["Date", "TIC", "Adj Close", "Open"]

✅ Up-to-date, using cache


---
# Backtest - Simulation

In [4]:
backtest_df, trade_blotter, portfolio_history = run_backtest(
    price_df=priceData,
    screen_rule=SCREEN,
    initial_cash=config["cash"],
    rebalance_freq=config["rebalance_frequency"],
    feature_df=load_full_data,
    start_date=start_date,    
    end_date=end_date,
    allocator="max_sharpe",
    ranker_dict=config["ranker"],
    config=config,
)


📅 Progress: 0/1546 | Date: 2020-01-01

📅 Progress: 10/1546 | Date: 2020-01-15

📅 Progress: 20/1546 | Date: 2020-01-29

🔁 Rebalance triggered on 2020-01-31
   💰 Portfolio value: 1,000,000.00
   🧾 Cash: 1,000,000.00
   🎯 Screened stocks: 100
   📊 Ranked stocks: 10
   🧠 Top allocations: [('BBOX.NS', 0.62795), ('EPL.NS', 0.13884), ('ALKYLAMINE.NS', 0.0868), ('RELAXO.NS', 0.06622), ('IGL.NS', 0.04987)]
   💼 Trades executed | New cash: 445.69

📅 Progress: 30/1546 | Date: 2020-02-12

📅 Progress: 40/1546 | Date: 2020-02-27

🔁 Rebalance triggered on 2020-02-28
   💰 Portfolio value: 1,436,706.85
   🧾 Cash: 445.69
   🎯 Screened stocks: 62
   📊 Ranked stocks: 10
   🧠 Top allocations: [('BBOX.NS', 0.65733), ('ATUL.NS', 0.24601), ('DIVISLAB.NS', 0.04924), ('ALKYLAMINE.NS', 0.03567), ('JBCHEPHARM.NS', 0.01175)]
   💼 Trades executed | New cash: 20,043.20

📅 Progress: 50/1546 | Date: 2020-03-13

📅 Progress: 60/1546 | Date: 2020-03-27

🔁 Rebalance triggered on 2020-03-31
   💰 Portfolio value: 924,724.4

---
# Portfolio Analysis

In [5]:
pv = backtest_df.copy()
pv["Date"] = pd.to_datetime(pv["Date"])
pv = pv.set_index("Date")["equity"]

bv = benchmark_prices.copy()
bv["Date"] = pd.to_datetime(bv["Date"])
bv = bv.set_index("Date")["bench"]

# align with portfolio dates
bv = bv.reindex(pv.index).ffill()

In [6]:
ts = Tearsheet(
    portfolio_values = pv,
    benchmark_values = bv,
    trades = None,   # ignore trades for now,
    risk_free_rate = 0.06
)

# All stats as Plotly tables (show inline in Jupyter/VSCode)
tables = ts.summary()
display(tables["summary"])

# Everything at once
ts.plot_all()

,Value
Start Date,2020-01-01
End Date,2026-04-02
N Days,1546.00
N Years,6.25
Total Return,687.47%
CAGR,39.12%
Volatility,35.88%
Sharpe,0.95
Sortino,1.43
Calmar,0.60



══════════════════════════════════════════════════════════════
  PORTFOLIO TEARSHEET
  2020-01-01  →  2026-04-02  (1546 days / 6.2 yrs)
══════════════════════════════════════════════════════════════

